# Capitolo 6 — Più variabili, più orizzonte: la serie di Jena (§ 6.5)
Dati gli ultimi 3 giorni di sei variabili orarie, prevedere la temperatura fra 24 ore. Circa 3 minuti.

In [ ]:
import sys; sys.path.insert(0, "..")
from utils import fissa_seme
import dati
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
fissa_seme(42)

df = dati.jena(); print(df.shape)
col = ["T (degC)", "p (mbar)", "rh (%)", "wv (m/s)", "sh (g/kg)", "Tdew (degC)"]
A = df[col].values.astype(np.float32); n = len(A); n_tr, n_va = int(n * 0.7), int(n * 0.85)
F, HOR = 72, 24
media, dev = A[:n_tr].mean(0), A[:n_tr].std(0); An = (A - media) / dev
def finestre(Z, a, b, solo_T):
    X = np.stack([Z[i:i + F, :1] if solo_T else Z[i:i + F] for i in range(a, b - F - HOR + 1, 3)])
    y = Z[a + F + HOR - 1:b:3, 0][:len(X)]
    return torch.from_numpy(X), torch.from_numpy(y)
mae = lambda a, b: float(np.mean(np.abs(a - b)))

In [ ]:
class M(nn.Module):
    def __init__(self, d): super().__init__(); self.l = nn.LSTM(d, 32, batch_first=True); self.fc = nn.Linear(32, 1)
    def forward(self, x): o, _ = self.l(x); return self.fc(o[:, -1]).squeeze(1)

risultati = {}
for solo_T in (True, False):
    Xtr, ytr = finestre(An, 0, n_tr, solo_T); Xva, yva = finestre(An, n_tr, n_va, solo_T); Xte, yte = finestre(An, n_va, n, solo_T)
    fissa_seme(42); m = M(Xtr.shape[2]); opt = torch.optim.Adam(m.parameters(), lr=1e-3); fn = nn.MSELoss(); migliore = (float("inf"), None)
    for epoca in range(15):
        m.train(); perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), 128):
            idx = perm[i:i + 128]; opt.zero_grad(); fn(m(Xtr[idx]), ytr[idx]).backward(); opt.step()
        m.eval()
        with torch.no_grad(): lv = fn(m(Xva), yva).item()
        if lv < migliore[0]: migliore = (lv, {k: v.clone() for k, v in m.state_dict().items()})
    m.load_state_dict(migliore[1]); m.eval()
    with torch.no_grad(): p = m(Xte).numpy() * dev[0] + media[0]
    yt = yte.numpy() * dev[0] + media[0]
    risultati["solo T" if solo_T else "6 variabili"] = mae(p, yt)
    if solo_T:
        risultati["persistenza (24 h fa)"] = mae(Xte[:, -1, 0].numpy() * dev[0] + media[0], yt)
        risultati["media ultime 24 h"] = mae(Xte[:, -24:, 0].numpy().mean(1) * dev[0] + media[0], yt)
for k, v in risultati.items(): print(f"{k:24s} MAE {v:.2f} °C")